# Parte 2: Exploración (EDA), Diccionario de Variables y Preprocesador

**Proyecto:** Predicción de Deserción Universitaria (Sistemas Inteligentes - UPN)  
**Objetivo de esta entrega (P2):**
1. Cargar el dataset crudo y limpiar encabezados (`str.strip()`).
2. Exploración (EDA) y auditoría de calidad de datos (0 nulos, 0 duplicados).
3. Clasificación rigurosa de las 36 variables (Nominales, Binarias y Numéricas).
4. Diccionario de variables completo.
5. Binarización del target y partición 80/20 estratificada con `random_state=42`.
6. Construcción del preprocesador (`ColumnTransformer`) ajustado **únicamente con train**.
7. Verificación del espacio de características y ausencia de fuga de datos (*data leakage*).

## 1. Carga del dataset original y sanitización de encabezados

In [ ]:
import sys
from pathlib import Path

# Asegurar que la raíz del proyecto esté en el sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
from src.config import DATA_RAW, TARGET_COL, POSITIVE_LABEL, SEED, TEST_SIZE

# Lectura con separador ';' y limpieza de espacios en columnas
df = pd.read_csv(DATA_RAW, sep=';')
df.columns = df.columns.str.strip()

print(f"Filas: {df.shape[0]} | Columnas totales: {df.shape[1]}")
df.head(3)

## 2. Exploración (EDA) y Documentación de Calidad
Comprobamos formalmente que el dataset no posee valores nulos ni duplicados, registrando la evidencia de calidad de origen.

In [ ]:
nulos_totales = df.isna().sum().sum()
duplicados = df.duplicated().sum()

print(f"Valores nulos totales en el dataset: {nulos_totales}")
print(f"Filas duplicadas en el dataset: {duplicados}")

print("\n--- Distribución de la variable Target original (3 clases) ---")
dist_target = pd.DataFrame({
    'Frecuencia': df[TARGET_COL].value_counts(),
    'Porcentaje (%)': (df[TARGET_COL].value_counts(normalize=True) * 100).round(2)
})
dist_target

## 3. Clasificación de las 36 Variables Predictoras
Es crítico separar las variables por su naturaleza semántica para evitar tratarlas erróneamente como números cardinales:
- **Nominales codificadas con números (9):** Variables donde los números son identificadores de categorías sin orden intrínseco (ej. Course=171 vs 9070). Se les aplicará `OneHotEncoder`.
- **Binarias (8):** Flags de 0 y 1. No requieren one-hot ni escalado; se dejan como `passthrough`.
- **Numéricas continuas y discretas de conteo (19):** Calificaciones, edades, conteos de unidades curriculares y tasas macroeconómicas. Se estandarizan con `StandardScaler`.

In [ ]:
from src.preprocessor import NOMINAL_COLS, BINARY_COLS, NUMERIC_COLS, ALL_FEATURE_COLS

print(f"Nominales (One-Hot)      : {len(NOMINAL_COLS)} variables")
print(f"Binarias (Passthrough)   : {len(BINARY_COLS)} variables")
print(f"Numéricas (StandardScaler): {len(NUMERIC_COLS)} variables")
print(f"Total variables clasificadas: {len(ALL_FEATURE_COLS)} de 36 predictoras")

assert len(ALL_FEATURE_COLS) == 36
assert set(ALL_FEATURE_COLS) | {TARGET_COL} == set(df.columns)

## 4. Diccionario Completo de Variables (36 Filas)

In [ ]:
diccionario_data = [
    # Nominales (9)
    ('Marital Status', 'Nominal (código numérico)', 'Estado civil del estudiante (1=Soltero, 2=Casado, etc.)', 'OneHotEncoder'),
    ('Application mode', 'Nominal (código numérico)', 'Modalidad de postulación / admisión a la universidad', 'OneHotEncoder'),
    ('Course', 'Nominal (código numérico)', 'Carrera profesional cursada por el estudiante', 'OneHotEncoder'),
    ('Previous qualification', 'Nominal (código numérico)', 'Nivel educativo previo del ingresante', 'OneHotEncoder'),
    ('Nacionality', 'Nominal (código numérico)', 'Nacionalidad de origen del estudiante', 'OneHotEncoder'),
    ("Mother's qualification", 'Nominal (código numérico)', 'Nivel de instrucción alcanzado por la madre', 'OneHotEncoder'),
    ("Father's qualification", 'Nominal (código numérico)', 'Nivel de instrucción alcanzado por el padre', 'OneHotEncoder'),
    ("Mother's occupation", 'Nominal (código numérico)', 'Ocupación laboral principal de la madre', 'OneHotEncoder'),
    ("Father's occupation", 'Nominal (código numérico)', 'Ocupación laboral principal del padre', 'OneHotEncoder'),
    # Binarias (8)
    ('Daytime/evening attendance', 'Binaria (0/1)', 'Turno de asistencia (1=Diurno, 0=Nocturno)', 'Passthrough (0/1 intacto)'),
    ('Displaced', 'Binaria (0/1)', 'Estudiante foráneo o desplazado de su localidad (1=Sí, 0=No)', 'Passthrough (0/1 intacto)'),
    ('Educational special needs', 'Binaria (0/1)', 'Presenta necesidades educativas especiales (1=Sí, 0=No)', 'Passthrough (0/1 intacto)'),
    ('Debtor', 'Binaria (0/1)', 'Tiene deudas o cuotas pendientes de pago (1=Sí, 0=No)', 'Passthrough (0/1 intacto)'),
    ('Tuition fees up to date', 'Binaria (0/1)', 'Al día con las tasas de matrícula (1=Sí, 0=No)', 'Passthrough (0/1 intacto)'),
    ('Gender', 'Binaria (0/1)', 'Género reportado (1=Masculino, 0=Femenino)', 'Passthrough (0/1 intacto)'),
    ('Scholarship holder', 'Binaria (0/1)', 'Beneficiario de beca de estudios (1=Sí, 0=No)', 'Passthrough (0/1 intacto)'),
    ('International', 'Binaria (0/1)', 'Estudiante internacional (1=Sí, 0=No)', 'Passthrough (0/1 intacto)'),
    # Numéricas (19)
    ('Application order', 'Numérica discreta', 'Preferencia de orden de carrera elegida (0 a 9)', 'StandardScaler'),
    ('Age at enrollment', 'Numérica continua', 'Edad del estudiante al momento de matricularse (17 a 70)', 'StandardScaler'),
    ('Previous qualification (grade)', 'Numérica continua', 'Calificación de educación previa (escala 0-200)', 'StandardScaler'),
    ('Admission grade', 'Numérica continua', 'Calificación del examen de ingreso (escala 0-200)', 'StandardScaler'),
    ('Curricular units 1st sem (credited)', 'Numérica discreta', 'Créditos convalidados en 1er semestre', 'StandardScaler'),
    ('Curricular units 1st sem (enrolled)', 'Numérica discreta', 'Cursos matriculados en 1er semestre', 'StandardScaler'),
    ('Curricular units 1st sem (evaluations)', 'Numérica discreta', 'Evaluaciones rendidas en 1er semestre', 'StandardScaler'),
    ('Curricular units 1st sem (approved)', 'Numérica discreta', 'Cursos aprobados en 1er semestre', 'StandardScaler'),
    ('Curricular units 1st sem (grade)', 'Numérica continua', 'Promedio de notas en 1er semestre (escala 0-20)', 'StandardScaler'),
    ('Curricular units 1st sem (without evaluations)', 'Numérica discreta', 'Cursos sin evaluación en 1er semestre', 'StandardScaler'),
    ('Curricular units 2nd sem (credited)', 'Numérica discreta', 'Créditos convalidados en 2do semestre', 'StandardScaler'),
    ('Curricular units 2nd sem (enrolled)', 'Numérica discreta', 'Cursos matriculados en 2do semestre', 'StandardScaler'),
    ('Curricular units 2nd sem (evaluations)', 'Numérica discreta', 'Evaluaciones rendidas en 2do semestre', 'StandardScaler'),
    ('Curricular units 2nd sem (approved)', 'Numérica discreta', 'Cursos aprobados en 2do semestre', 'StandardScaler'),
    ('Curricular units 2nd sem (grade)', 'Numérica continua', 'Promedio de notas en 2do semestre (escala 0-20)', 'StandardScaler'),
    ('Curricular units 2nd sem (without evaluations)', 'Numérica discreta', 'Cursos sin evaluación en 2do semestre', 'StandardScaler'),
    ('Unemployment rate', 'Numérica continua', 'Tasa de desempleo del país (%)', 'StandardScaler'),
    ('Inflation rate', 'Numérica continua', 'Tasa de inflación anual (%)', 'StandardScaler'),
    ('GDP', 'Numérica continua', 'Tasa de crecimiento del PBI (%)', 'StandardScaler')
]

df_diccionario = pd.DataFrame(diccionario_data, columns=['Variable', 'Tipo de Variable', 'Descripción', 'Tratamiento'])
pd.set_option('display.max_rows', 40)
df_diccionario

## 5. Objetivo Binario y Partición Estratificada 80/20

In [ ]:
from sklearn.model_selection import train_test_split

# Objetivo binario acordado: 1 = Dropout, 0 = Enrolled + Graduate
y = (df[TARGET_COL] == POSITIVE_LABEL).astype(int).rename('dropout')
X = df[ALL_FEATURE_COLS]

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}  | y_test : {y_test.shape}")
print(f"Proporción de Dropout en Train: {y_train.mean()*100:.2f} % ({y_train.sum()} alumnos)")
print(f"Proporción de Dropout en Test : {y_test.mean()*100:.2f} % ({y_test.sum()} alumnos)")

# Comprobaciones de integridad
assert len(X_train) == 3539 and len(X_test) == 885
assert abs(y_train.mean() - y_test.mean()) < 0.005
assert not set(X_train.index) & set(X_test.index), 'Fuga: índices solapados'

## 6. Construcción y Ajuste del Preprocesador (`ColumnTransformer`)
**Regla fundamental de Ciencia de Datos:** El preprocesador se ajusta (`fit`) **exclusivamente sobre `X_train`** para no filtrar información del conjunto de prueba (*Data Leakage*).

In [ ]:
from src.preprocessor import get_preprocessor

preprocessor = get_preprocessor()

# Ajustar SOLO en train
preprocessor.fit(X_train)

# Transformar ambos conjuntos
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

columnas_finales = preprocessor.get_feature_names_out()
print(f"Columnas originales : {X_train.shape[1]}")
print(f"Columnas expandidas : {len(columnas_finales)} (tras OneHotEncoder)")
print(f"Forma de X_train_trans: {X_train_trans.shape}")
print(f"Forma de X_test_trans : {X_test_trans.shape}")

## 7. Verificación de Integridad Antes y Después
Comprobamos que las matrices no presenten ningún valor nulo tras la transformación y mostramos una muestra comparativa.

In [ ]:
# Verificación de nulos
assert np.isnan(X_train_trans).sum() == 0, 'Existen NaN en train transformado'
assert np.isnan(X_test_trans).sum() == 0, 'Existen NaN en test transformado'
assert X_train_trans.shape[1] == X_test_trans.shape[1], 'Discrepancia en número de columnas'

print("✔ Verificación exitosa: 0 nulos y dimensiones coincidentes en ambos sets.")

# Visualizar las primeras filas transformadas en un DataFrame legible
df_train_trans = pd.DataFrame(X_train_trans, columns=columnas_finales)
print("\nMuestra del conjunto de entrenamiento transformado (primeras 5 columnas):")
display(df_train_trans.iloc[:3, :5])

print("\nMuestra de columnas escaladas continuas (ej. notas y edad):")
display(df_train_trans[['Age at enrollment', 'Admission grade', 'Curricular units 1st sem (grade)']].head(3))